# Accessing the European Nucleotide Archive (ENA) with BioServices

The **European Nucleotide Archive (ENA)** is a comprehensive, freely accessible repository of nucleotide sequencing information maintained by EMBL-EBI. It stores:

- **Raw sequencing reads** (from next-generation and long-read platforms)
- **Sequence assemblies** (assembled genomes, transcriptomes)
- **Annotated sequences** (coding sequences, non-coding RNAs, complete genomes)

Data in ENA spans all domains of life and is organised into a hierarchy of accession types:
- `ERP`, `SRP`, `DRP` — study/project records
- `ERX`, `SRX`, `DRX` — experiment records
- `ERR`, `SRR`, `DRR` — run records (raw reads)
- `ERA`, `ERS` — submission/sample records
- `A00001`-style alphanumeric IDs — assembled/annotated sequences

BioServices provides the `ENA` class that wraps the ENA browser API. The primary method is `get_data(identifier, frmt)` which retrieves a record in a variety of formats.

In [ ]:
from bioservices import ENA

e = ENA()

---
## 1. Retrieving metadata in XML format

XML is the most information-rich format for ENA records. It contains the full metadata tree for a given accession, including experiment design, library strategy, instrument model, and links to associated records.

Here we retrieve the metadata for **ERA000092**, an early ENA experiment record.

In [ ]:
xml_data = e.get_data('ERA000092', 'xml')
# Print the first 800 characters of the XML response
print(xml_data[:800])

---
## 2. Retrieving an annotated sequence in FASTA format

Assembled and annotated sequences can be retrieved directly in FASTA format. **A00145** is a classic ENA entry: the *B. taurus* (bovine) interferon-alpha A mRNA sequence, deposited in the earliest days of the EMBL Nucleotide Sequence Database.

In [ ]:
fasta = e.get_data('A00145', 'fasta')
print(fasta)

### 2.1 Retrieving a subsequence with `fasta_range`

The `fasta_range` parameter asks the ENA API to return only a slice of the sequence, specified as `[start, end]` with 1-based coordinates. This is useful when you only need a particular region (e.g. a coding sequence within a larger record) without downloading the full sequence.

> **Note:** The ENA service may return the full sequence for short records or if the requested range encompasses the whole sequence. The header line in the FASTA output will still show the original accession.

In [ ]:
# Request only nucleotides 1–60 of A00145
sub_fasta = e.get_data('A00145', 'fasta', fasta_range=[1, 60])
print('Subsequence (positions 1-60):')
print(sub_fasta)

---
## 3. Retrieving a sequence in EMBL flat-file format

The EMBL flat-file (`embl`) format is the native format for EMBL/ENA sequence records. It contains rich annotation including feature tables (CDS, exon, UTR, etc.) with qualifier fields. This is equivalent to the GenBank flat-file format used by NCBI.

In [ ]:
embl_text = e.get_data('A00145', 'text')
# Print the first 1000 characters to inspect the header and feature table
print(embl_text[:1000])

---
## 4. Retrieving raw read data

ENA run records (accessions starting with `ERR`, `SRR`, or `DRR`) hold the raw sequencing reads from high-throughput experiments. These can be retrieved in **FASTQ** format.

> **Note:** FASTQ files from real run accessions can be very large. The example below shows the structure of a FASTQ request; the data size depends on the run.

For sequence accessions it is also straightforward to retrieve the record in **HTML** format, which renders the record as a browser page:


In [ ]:
# Retrieve the A00145 record as a human-readable HTML page
# (the result is an HTML string that can be rendered in a browser or Jupyter)
html_data = e.get_data('A00145', 'html')
# Show just the first 500 characters of the HTML
print(html_data[:500])

---
## 5. Format summary

The `get_data()` method supports a range of output formats depending on the type of accession being queried:

| Format | Description | Suitable for |
|--------|-------------|-------------|
| `xml`  | Full XML metadata tree | All accession types |
| `fasta` | Nucleotide/protein sequence in FASTA | Sequence accessions |
| `fastq` | Raw read sequences with quality scores | Run accessions (SRR/ERR/DRR) |
| `text`  | EMBL flat-file with feature annotations | Sequence accessions |
| `embl`  | Alias for `text` flat-file format | Sequence accessions |
| `html`  | Human-readable browser view | All accession types |

Not all formats are available for all accession types. For example, FASTQ is only meaningful for raw sequencing run records, while EMBL flat-file format is specific to annotated sequence records.